# A10. 중심극한정리 (Central Limit Theorem)

**어떤 분포에서든, 표본 크기가 충분히 크면 표본평균은 정규분포에 수렴한다.**

$$
\bar{X}_n = \frac{1}{n}\sum_{i=1}^{n} X_i \xrightarrow{d} N\left(\mu, \frac{\sigma^2}{n}\right)
$$

| 기호 | 의미 |
|:---:|:---|
| $\bar{X}_n$ | $n$개 표본의 평균 |
| $\mu$ | 모집단 평균 |
| $\sigma^2$ | 모집단 분산 |
| $n$ | 표본 크기 |

> **CLT의 놀라운 점**: 원래 분포가 균등이든, 지수이든, 심지어 이상치가 있든 — $n$이 충분히 크면 **표본평균은 항상 정규분포**가 된다!

In [ ]:
# === 공통 설정 ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from scipy import stats

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')

# 재현성
np.random.seed(42)

print("라이브러리 로드 완료")

---
## 시뮬레이션 1: 주사위 CLT

주사위 1개의 결과는 **균등분포(Uniform)** — 1~6이 각각 1/6 확률이다.  
주사위를 여러 개 던져 **합**을 구하면, 그 분포는 어떻게 변할까?

$$
S_n = \sum_{i=1}^{n} X_i, \quad X_i \sim \text{Uniform}\{1, 2, 3, 4, 5, 6\}
$$

| 주사위 수 | 이론 평균 | 이론 표준편차 |
|:---:|:---:|:---:|
| $n$ | $3.5n$ | $1.708\sqrt{n}$ |

In [ ]:
# 주사위 시뮬레이션
dice_counts = [1, 2, 3, 10, 25, 50]
n_simulations = 1000

# 주사위 1개의 이론값
mu_one = 3.5        # E[X] = (1+2+3+4+5+6)/6
var_one = 35/12     # Var[X] = E[X^2] - (E[X])^2
sigma_one = np.sqrt(var_one)

print(f"주사위 1개: 평균 = {mu_one}, 분산 = {var_one:.4f}, 표준편차 = {sigma_one:.4f}")
print(f"\n각 주사위 수별 {n_simulations}회 시뮬레이션 수행...")

# 시뮬레이션 결과 저장
results = {}
for n_dice in dice_counts:
    sums = np.sum(np.random.randint(1, 7, size=(n_simulations, n_dice)), axis=1)
    results[n_dice] = sums
    theory_mean = mu_one * n_dice
    theory_std = sigma_one * np.sqrt(n_dice)
    print(f"  주사위 {n_dice:2d}개: 실측평균={sums.mean():.2f} (이론={theory_mean:.2f}), "
          f"실측표준편차={sums.std():.2f} (이론={theory_std:.2f})")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

colors = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#3498db', '#9b59b6']

for idx, (n_dice, color) in enumerate(zip(dice_counts, colors)):
    ax = axes[idx]
    sums = results[n_dice]
    
    # 히스토그램
    ax.hist(sums, bins=min(30, n_dice * 6), density=True,
            color=color, edgecolor='white', alpha=0.7, label='시뮬레이션')
    
    # 이론 정규 곡선 오버레이
    theory_mean = mu_one * n_dice
    theory_std = sigma_one * np.sqrt(n_dice)
    x_range = np.linspace(sums.min(), sums.max(), 200)
    normal_pdf = stats.norm.pdf(x_range, loc=theory_mean, scale=theory_std)
    ax.plot(x_range, normal_pdf, 'k-', linewidth=2.5, label='이론 정규분포')
    
    ax.set_title(f'주사위 {n_dice}개의 합', fontsize=14, fontweight='bold')
    ax.set_xlabel('합', fontsize=11)
    ax.set_ylabel('확률밀도', fontsize=11)
    ax.legend(fontsize=9)
    
    # 정보 텍스트
    textstr = f'$\\mu={theory_mean:.1f}$\n$\\sigma={theory_std:.2f}$'
    ax.text(0.95, 0.95, textstr, transform=ax.transAxes,
            fontsize=10, verticalalignment='top', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

fig.suptitle('균등분포 → 정규분포: 주사위 합의 CLT 시각적 증명',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 해석

- 주사위 **1개**: 균등분포 (평평한 막대)
- 주사위 **2개**: 삼각형 모양 (가운데가 높아지기 시작)
- 주사위 **3개**: 종 모양이 나타나기 시작
- 주사위 **10개 이상**: 거의 완벽한 **정규분포**!

> **결론**: 균등분포라는 전혀 정규분포가 아닌 분포에서 시작했지만,  
> 합(또는 평균)을 구하면 **정규분포로 수렴**한다. 이것이 CLT의 핵심이다.

---
## 시뮬레이션 2: 다양한 원본 분포에서의 CLT

CLT가 정말 **어떤 분포에서든** 작동하는지 확인해 보자.

| 분포 | 특징 | 모양 |
|:---|:---|:---|
| Uniform(0,1) | 완전히 평평 | 직사각형 |
| Exponential(1) | 강한 우편향 | 급격한 감소 |
| Pareto(2.5) | 극단적 우편향 (Heavy Tail) | 매우 긴 꼬리 |
| With Outliers | 정규+이상치 혼합 | 중심 + 튀는 값 |

In [ ]:
np.random.seed(42)
n_samples_list = [30, 50]  # 표본 크기
n_reps = 1000              # 반복 횟수
pop_size = 100_000         # 모집단 크기

# 4가지 모집단 생성
populations = {
    'Uniform(0,1)': np.random.uniform(0, 1, pop_size),
    'Exponential(1)': np.random.exponential(1, pop_size),
    'Pareto(2.5)': stats.pareto.rvs(b=2.5, size=pop_size),
    'With Outliers': np.concatenate([
        np.random.normal(50, 10, int(pop_size * 0.95)),
        np.random.normal(200, 30, int(pop_size * 0.05))  # 5% 이상치
    ])
}

for name, pop in populations.items():
    print(f"{name:20s}: 평균={pop.mean():.3f}, 표준편차={pop.std():.3f}, "
          f"왜도={stats.skew(pop):.3f}")

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(20, 13))

dist_colors = ['#3498db', '#e74c3c', '#2ecc71', '#9b59b6']

for col, ((name, pop), color) in enumerate(zip(populations.items(), dist_colors)):
    
    # 행 0: 원본 분포
    ax = axes[0][col]
    ax.hist(pop, bins=50, density=True, color=color, edgecolor='white', alpha=0.7)
    ax.set_title(f'원본: {name}', fontsize=13, fontweight='bold')
    ax.set_ylabel('확률밀도' if col == 0 else '', fontsize=11)
    if name == 'Pareto(2.5)':
        ax.set_xlim(0, 10)  # Pareto 꼬리 잘라서 표시
    
    # 행 1: n=30 표본평균
    for row, n_samp in enumerate(n_samples_list, 1):
        ax = axes[row][col]
        sample_means = [np.mean(np.random.choice(pop, size=n_samp, replace=False))
                        for _ in range(n_reps)]
        sample_means = np.array(sample_means)
        
        ax.hist(sample_means, bins=30, density=True,
                color=color, edgecolor='white', alpha=0.6)
        
        # 이론 정규 곡선
        x_range = np.linspace(sample_means.min(), sample_means.max(), 200)
        normal_pdf = stats.norm.pdf(x_range,
                                     loc=sample_means.mean(),
                                     scale=sample_means.std())
        ax.plot(x_range, normal_pdf, 'k-', linewidth=2)
        
        ax.set_title(f'표본평균 (n={n_samp})', fontsize=12)
        ax.set_ylabel('확률밀도' if col == 0 else '', fontsize=11)
        
        # Shapiro-Wilk 정규성 검정
        _, p_val = stats.shapiro(sample_means[:500])  # 500개까지만
        normal_str = '정규!' if p_val > 0.05 else f'p={p_val:.3f}'
        ax.text(0.95, 0.95, normal_str, transform=ax.transAxes,
                fontsize=10, ha='right', va='top',
                color='green' if p_val > 0.05 else 'orange',
                fontweight='bold',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# 행 라벨
axes[0][0].set_ylabel('원본 분포\n확률밀도', fontsize=12, fontweight='bold')
axes[1][0].set_ylabel('n=30 표본평균\n확률밀도', fontsize=12, fontweight='bold')
axes[2][0].set_ylabel('n=50 표본평균\n확률밀도', fontsize=12, fontweight='bold')

fig.suptitle('CLT: 어떤 분포에서든 표본평균은 정규분포로 수렴한다',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 해석

- **첫째 행** (원본): 각각 완전히 다른 모양 — 평평, 편향, 극단적 꼬리, 이상치
- **둘째/셋째 행** (표본평균): n=30~50이면 **모두 정규분포 모양**으로 수렴!
- Pareto(Heavy Tail)도 n=50이면 거의 정규분포가 된다
- 이상치가 있어도 표본평균의 분포는 안정적이다

> **핵심**: CLT 덕분에 우리는 **원본 분포를 몰라도** 표본평균에 대해  
> 정규분포 기반의 통계적 추론(신뢰구간, 가설검정)을 할 수 있다!

---
## 시뮬레이션 3: Diamonds Price 표본평균

실제 데이터로 CLT를 확인하자.  
**diamonds** 데이터셋의 가격(price)은 **강한 우편향** 분포이다.  
여기서 50개씩 비복원추출하여 표본평균을 500번 구하면?

In [ ]:
# Diamonds 데이터 로드
diamonds = sns.load_dataset('diamonds')
prices = diamonds['price'].values

print(f"다이아몬드 총 개수: {len(prices):,}")
print(f"가격 평균: ${prices.mean():,.2f}")
print(f"가격 중앙값: ${np.median(prices):,.2f}")
print(f"가격 표준편차: ${prices.std():,.2f}")
print(f"왜도(Skewness): {stats.skew(prices):.3f}")
print(f"첨도(Kurtosis): {stats.kurtosis(prices):.3f}")

In [ ]:
# 비복원추출로 표본평균 생성
np.random.seed(42)
sample_size = 50
n_iterations = 500

sample_means = np.array([
    np.mean(np.random.choice(prices, size=sample_size, replace=False))
    for _ in range(n_iterations)
])

print(f"표본 크기: {sample_size}")
print(f"반복 횟수: {n_iterations}")
print(f"\n표본평균들의 평균: ${sample_means.mean():,.2f}")
print(f"표본평균들의 표준편차: ${sample_means.std():,.2f}")
print(f"\nCLT 이론 표준오차: ${prices.std() / np.sqrt(sample_size):,.2f}")
print(f"표본평균 왜도: {stats.skew(sample_means):.3f} (원본: {stats.skew(prices):.3f})")

# 정규성 검정
stat_sw, p_sw = stats.shapiro(sample_means)
print(f"\nShapiro-Wilk 정규성 검정: p-value = {p_sw:.4f}")
print(f"→ {'정규분포로 볼 수 있다 (p > 0.05)' if p_sw > 0.05 else '정규분포가 아니다 (p <= 0.05)'}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# (1) 원본 가격 분포
ax = axes[0]
ax.hist(prices, bins=60, density=True, color='#e74c3c', edgecolor='white', alpha=0.7)
ax.axvline(prices.mean(), color='blue', linestyle='--', linewidth=2,
           label=f'평균: ${prices.mean():,.0f}')
ax.axvline(np.median(prices), color='green', linestyle='--', linewidth=2,
           label=f'중앙값: ${np.median(prices):,.0f}')
ax.set_xlabel('가격 ($)', fontsize=12)
ax.set_ylabel('확률밀도', fontsize=12)
ax.set_title('원본: 다이아몬드 가격 분포\n(강한 우편향)', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)

# (2) 표본평균 분포
ax = axes[1]
ax.hist(sample_means, bins=30, density=True, color='#3498db', edgecolor='white', alpha=0.7,
        label='표본평균 분포')

# 이론 정규 곡선 오버레이
x_range = np.linspace(sample_means.min(), sample_means.max(), 200)
normal_pdf = stats.norm.pdf(x_range, loc=sample_means.mean(), scale=sample_means.std())
ax.plot(x_range, normal_pdf, 'k-', linewidth=2.5, label='정규분포 곡선')

ax.axvline(sample_means.mean(), color='blue', linestyle='--', linewidth=2,
           label=f'평균: ${sample_means.mean():,.0f}')
ax.set_xlabel('표본평균 ($)', fontsize=12)
ax.set_ylabel('확률밀도', fontsize=12)
ax.set_title(f'표본평균 분포 (n={sample_size}, {n_iterations}회)\n→ 정규분포!',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=10)

# 화살표 연결
fig.text(0.5, 0.5, 'CLT', fontsize=20, fontweight='bold',
         ha='center', va='center', color='#2c3e50',
         bbox=dict(boxstyle='rarrow,pad=0.3', facecolor='#f39c12', alpha=0.8))

plt.tight_layout()
plt.show()

### 해석

- **원본** 가격 분포는 강한 우편향 — 정규분포와 거리가 멀다
- **표본평균** 분포는 거의 완벽한 정규분포! (Shapiro-Wilk 검정으로 확인)
- 원본의 왜도가 크더라도, 50개씩 뽑아 평균을 내면 **왜도가 거의 0**에 수렴
- 표본평균의 표준편차가 원본보다 **훨씬 작다** → $\sigma / \sqrt{n}$ 효과

---
## 토론 1: 왜 서로 다른 분포인데 결과는 같아지는가?

### 직관적 이해

$$
\bar{X} = \frac{X_1 + X_2 + \cdots + X_n}{n}
$$

평균을 구하는 과정에서:

1. **극단값의 상쇄**: 매우 큰 값과 매우 작은 값이 서로 상쇄된다
2. **변동성 감소**: $n$이 커질수록 개별 관측치의 영향력이 $1/n$로 줄어든다
3. **중심 집중**: 평균들은 모평균 근처에 점점 더 집중된다

### 수학적 근거

$$
\text{Var}(\bar{X}) = \frac{\sigma^2}{n} \xrightarrow{n \to \infty} 0
$$

- 표본 크기 $n$이 커질수록 표본평균의 분산은 **0에 수렴**
- 분산이 줄어드는 속도는 원본 분포에 무관하다 (유한 분산만 필요)

---
## 토론 2: 이상치가 있어도 평균은 왜 안정적인가?

### 핵심 논리

이상치가 있어도 CLT가 작동하는 이유:

1. **대수의 법칙**: $n$이 커지면 이상치의 **비율적 영향**이 줄어든다
   - 이상치 1개 / 전체 50개 = 2% 영향
   - 이상치 1개 / 전체 500개 = 0.2% 영향

2. **독립성**: 각 표본이 독립적으로 추출되므로, 한 표본에 이상치가 포함되어도 다른 표본에는 영향 없음

3. **반복 효과**: 500번의 표본 추출 중 이상치가 포함되는 경우와 아닌 경우가 섞여 **평균적으로 상쇄**

### 주의: CLT의 한계

- **분산이 무한**인 분포(예: Cauchy)에서는 CLT가 작동하지 않는다!
- 표본 크기가 **매우 작으면**(n < 5) 정규 근사가 부정확할 수 있다
- Heavy-tail 분포는 더 **큰 n**이 필요하다

In [ ]:
# CLT가 실패하는 경우: Cauchy 분포 (분산 = 무한)
np.random.seed(42)
n_test = 50
n_reps_test = 1000

# 정상 케이스: 정규분포
normal_means = [np.mean(np.random.normal(0, 1, n_test)) for _ in range(n_reps_test)]

# 실패 케이스: 코시 분포 (분산 = 무한)
cauchy_means = [np.mean(stats.cauchy.rvs(size=n_test)) for _ in range(n_reps_test)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.hist(normal_means, bins=30, density=True, color='#3498db', edgecolor='white', alpha=0.7)
ax.set_title(f'정규분포에서 표본평균 (n={n_test})\n→ CLT 성공!', fontsize=13, fontweight='bold')
ax.set_xlabel('표본평균', fontsize=11)
ax.set_ylabel('확률밀도', fontsize=11)

ax = axes[1]
cauchy_clipped = np.clip(cauchy_means, -20, 20)  # 시각화를 위해 클리핑
ax.hist(cauchy_clipped, bins=50, density=True, color='#e74c3c', edgecolor='white', alpha=0.7)
ax.set_title(f'코시분포에서 표본평균 (n={n_test})\n→ CLT 실패! (분산=무한)', fontsize=13, fontweight='bold')
ax.set_xlabel('표본평균 (클리핑됨)', fontsize=11)
ax.set_ylabel('확률밀도', fontsize=11)

plt.tight_layout()
plt.show()

print(f"정규분포 표본평균의 표준편차: {np.std(normal_means):.4f}")
print(f"코시분포 표본평균의 표준편차: {np.std(cauchy_means):.4f} (불안정!)")

---
## 토론 3: ML / 딥러닝 / A/B 테스트와의 관계

### 1. A/B 테스트

$$
H_0: \mu_A = \mu_B \quad \text{vs} \quad H_1: \mu_A \neq \mu_B
$$

- 실험군(A)과 대조군(B)의 **표본평균 차이**가 정규분포를 따른다고 가정 → **CLT 덕분!**
- 충분한 표본 크기만 확보하면 원본 데이터 분포에 상관없이 **z-검정, t-검정** 사용 가능

### 2. 딥러닝

- **미니배치 SGD**: 미니배치의 평균 그래디언트는 CLT에 의해 안정적
- 배치 크기가 클수록 그래디언트 추정이 안정 → 하지만 일반화 성능은 떨어질 수 있음
- **Batch Normalization**: 미니배치 통계량(평균, 분산)이 안정적인 이유 = CLT

### 3. 앙상블 학습

- Random Forest: 여러 트리 예측의 **평균** → CLT에 의해 개별 트리보다 안정적
- Bagging: 분산 감소 효과 $\text{Var}(\bar{f}) = \frac{\sigma^2}{n}$

### 4. 신뢰구간

$$
\bar{X} \pm z_{\alpha/2} \cdot \frac{s}{\sqrt{n}}
$$

이 공식이 성립하는 이유? **CLT** 때문이다!

---
## 종합 정리

### CLT의 3가지 핵심

| | 내용 |
|:---|:---|
| **보편성** | 원본이 어떤 분포든 (유한 분산이면) 표본평균은 정규분포 |
| **수렴 속도** | n=30이면 대부분 충분, Heavy-tail은 n=50+ 필요 |
| **실용적 의미** | 통계적 추론의 기초 — 신뢰구간, 가설검정, A/B 테스트 모두 CLT 위에 구축 |

### CLT가 없는 세상

- 모든 통계 검정에서 원본 분포를 정확히 알아야 한다
- A/B 테스트를 할 수 없다
- 여론조사 결과에 오차범위를 붙일 수 없다
- 딥러닝의 미니배치 학습이 불안정해진다

> **CLT는 통계학의 가장 강력한 정리이자, 현대 데이터 과학의 기반이다.**